# 02 - Data Cleaning

Notebook ini membersihkan dataset IMDb berdasarkan temuan pada tahap **data understanding**.

**Masalah yang ditemukan pada notebook 01:**

| No | Temuan | Keterangan |
|----|--------|------------|
| 1 | Kolom `Rank` | Hanya nomor urut (ID), tidak informatif untuk analisis |
| 2 | Missing value `Revenue (Millions)` | 128 data kosong |
| 3 | Missing value `Metascore` | 64 data kosong |
| 4 | Nilai `Revenue (Millions)` = 0 | Tidak wajar, kemungkinan data hilang |
| 5 | Outlier `Revenue (Millions)` | 55 film jauh di atas batas wajar (IQR) |
| 6 | Kolom teks majemuk | `Genre` dan `Actors` berisi beberapa nilai dalam satu sel |

**Tahapan cleaning:**

1. Menghapus kolom yang tidak digunakan.
2. Menangani nilai pendapatan yang tidak wajar.
3. Menangani outlier pendapatan (winsorization).
4. Imputasi missing value.
5. Merapikan teks.
6. Verifikasi dan menyimpan dataset bersih.

## 1. Import Library

Mengimpor `pandas` untuk proses manipulasi data.

In [ ]:
# Mengimpor library pandas untuk manipulasi data
import pandas as pd

## 2. Load Dataset

Membaca kembali dataset mentah yang sama seperti pada notebook 01.

In [ ]:
# Membaca dataset mentah ke dalam DataFrame
df = pd.read_csv("../data/imdb_movie_dataset.csv")

# Menampilkan ukuran data sebelum dibersihkan
df.shape

## 3. Melihat Kondisi Awal Data

Mengecek kembali missing value dan duplikat sebelum dilakukan pembersihan.

In [ ]:
# Menampilkan jumlah missing value pada setiap kolom
df.isnull().sum()

In [ ]:
# Menampilkan jumlah baris duplikat
df.duplicated().sum()

## 4. Menghapus Kolom yang Tidak Digunakan

Kolom `Rank` hanya berisi nomor urut baris (1-1000) sehingga tidak memiliki nilai analitis dan dihapus.

In [ ]:
# Menghapus kolom Rank karena hanya berisi nomor urut (ID)
df = df.drop(columns=["Rank"])

# Menampilkan kolom yang tersisa
df.columns

## 5. Menangani Nilai Tidak Wajar pada Revenue

Terdapat 1 film dengan pendapatan **0 juta**. Nilai ini tidak realistis untuk film yang masuk box office, sehingga dianggap sebagai data hilang dan diubah menjadi `NaN` agar ikut ditangani pada tahap imputasi.

In [ ]:
# Mengecek jumlah film dengan pendapatan 0
print("Jumlah film dengan Revenue = 0 :", (df["Revenue (Millions)"] == 0).sum())

# Mengubah nilai 0 menjadi NaN (dianggap data hilang)
df["Revenue (Millions)"] = df["Revenue (Millions)"].replace(0, float("nan"))

# Mengecek jumlah missing value Revenue setelah konversi
print("Jumlah missing value Revenue  :", df["Revenue (Millions)"].isnull().sum())

## 6. Menangani Outlier Revenue (Winsorization)

Pendapatan film sangat timpang: sebagian kecil film berpendapatan sangat besar. Outlier dideteksi dengan **metode IQR**, lalu nilai yang melewati batas atas dipotong (*capping*) ke batas tersebut agar tidak mendominasi analisis, tanpa menghapus data filmnya.

In [ ]:
# Menghitung batas wajar pendapatan dengan metode IQR
Q1 = df["Revenue (Millions)"].quantile(0.25)
Q3 = df["Revenue (Millions)"].quantile(0.75)
IQR = Q3 - Q1

batas_bawah = Q1 - 1.5 * IQR
batas_atas = Q3 + 1.5 * IQR

print(f"Q1         : {Q1:.2f}")
print(f"Q3         : {Q3:.2f}")
print(f"IQR        : {IQR:.2f}")
print(f"Batas bawah: {batas_bawah:.2f}")
print(f"Batas atas : {batas_atas:.2f}")
print("Jumlah outlier di atas batas atas:",
      (df["Revenue (Millions)"] > batas_atas).sum(), "film")

In [ ]:
# Winsorization: nilai di atas batas atas dipotong ke nilai batas atas
df["Revenue (Millions)"] = df["Revenue (Millions)"].clip(upper=batas_atas)

# Mengecek nilai maksimum setelah pemotongan
print(f"Nilai maksimum Revenue setelah capping: {df['Revenue (Millions)'].max():.2f}")

## 7. Imputasi Missing Value

Missing value diisi dengan **median** (bukan mean) karena median lebih tahan terhadap nilai ekstrem.

- `Revenue (Millions)`: 129 data kosong (128 kosong asli + 1 nilai 0).
- `Metascore`: 64 data kosong.

In [ ]:
# Menghitung nilai median sebagai pengganti data kosong
median_revenue = df["Revenue (Millions)"].median()
median_metascore = df["Metascore"].median()

# Mengisi missing value dengan nilai median
df["Revenue (Millions)"] = df["Revenue (Millions)"].fillna(median_revenue)
df["Metascore"] = df["Metascore"].fillna(median_metascore)

print(f"Revenue diisi dengan median   : {median_revenue:.2f}")
print(f"Metascore diisi dengan median : {median_metascore:.2f}")
print("Total missing value sekarang  :", df.isnull().sum().sum())

## 8. Merapikan Teks

Menghapus spasi berlebih di awal/akhir teks pada kolom bertipe teks agar nilai kategorik konsisten (misalnya `"Action, Adventure"` dan `"Action,Adventure"` dianggap sama).

In [ ]:
# Menghapus spasi berlebih pada kolom teks
kolom_teks = ["Title", "Genre", "Description", "Director", "Actors"]

for kolom in kolom_teks:
    df[kolom] = df[kolom].str.strip()

# Menampilkan contoh hasil pembersihan teks
df[kolom_teks].head(3)

## 9. Verifikasi Hasil Cleaning

Memastikan dataset sudah bersih: tidak ada missing value, tidak ada duplikat, dan tipe data sesuai.

In [ ]:
# Verifikasi akhir hasil cleaning
print("Ukuran data     :", df.shape)
print("Total missing   :", df.isnull().sum().sum())
print("Jumlah duplikat :", df.duplicated().sum())
print()
df.info()

In [ ]:
# Melihat statistik deskriptif setelah cleaning
df.describe().round(2)

## 10. Menyimpan Dataset Bersih

Dataset hasil cleaning disimpan agar dapat digunakan pada notebook EDA dan visualisasi.

In [ ]:
# Menyimpan dataset bersih ke file CSV baru
df.to_csv("../data/imdb_movie_dataset_clean.csv", index=False)

print("Dataset bersih tersimpan di: ../data/imdb_movie_dataset_clean.csv")
print("Ukuran akhir:", df.shape)

## 11. Kesimpulan Data Cleaning

- Kolom `Rank` dihapus karena hanya berupa nomor urut.
- Dataset akhir berukuran **1000 baris x 11 kolom**.
- Missing value `Revenue (Millions)` (129 data) dan `Metascore` (64 data) diisi dengan **median**.
- Nilai `Revenue (Millions)` = 0 dianggap data hilang dan ikut diimputasi.
- Outlier pendapatan dipotong ke batas atas IQR sehingga nilai maksimum turun dari 936,63 menjadi sekitar 264,28 juta.
- Tidak ada baris duplikat, dan seluruh kolom teks sudah dirapikan.
- Dataset siap digunakan untuk tahap **Exploratory Data Analysis** pada notebook `03_exploratory_data_analysis.ipynb`.